In [1]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col 
from pyspark.sql import functions as F

In [2]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [3]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [4]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 14:40:26 WARN Utils: Your hostname, harpar, resolves to a loopback address: 127.0.1.1; using 192.168.0.2 instead (on interface enp0s31f6)
26/05/06 14:40:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harpar/DEV/DataEngineering/FinalELTProject/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harpar/.ivy2.5.2/cache
The jars for the packages stored in: /home/harpar/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3997f5cf-80ca-431c-8065-a58e2bfc05d0;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
:: resolution report :: resolve 104ms :: artifacts dl 3ms
	:: modules in use:
	org.che

In [5]:
# Récupère la table windturbinepower_enriched depuis PostgreSQL dans un DataFrame Spark
windturbinepower_enriched_df = spark.read.jdbc(
    url=JDBC_URL,
    table='silver.windturbinepower_enriched',
    properties=JDBC_PROPS
)



In [6]:
# Renommer le DataFrame 
df = windturbinepower_enriched_df

# CRÉATION DES TABLES DE DIMENSION avec CLÉS STABLES

# 1. Date Dimension Table (date_dim)
# Clé naturelle : la date elle-même
date_dim = df.select("date", "day", "month", "quarter", "year").distinct() \
    .withColumnRenamed("date", "date_id") \
    .select("date_id", "day", "month", "quarter", "year")

# 2. Time Dimension Table (time_dim)
# Clé naturelle : l'heure elle-même
time_dim = df.select("time", "hour_of_day", "minute_of_hour", "second_of_minute", "time_period").distinct() \
    .withColumnRenamed("time", "time_id") \
    .select("time_id", "hour_of_day", "minute_of_hour", "second_of_minute", "time_period")

# 3. Turbine Dimension Table (turbine_dim)
# Clé stable : hash des attributs naturels (turbine_name + region)
turbine_dim = df.select("turbine_name", "capacity", "latitude", "longitude", "region", "region_name").distinct() \
    .withColumn("turbine_id", 
        F.abs(F.hash(F.concat_ws("_", F.col("turbine_name"), F.col("region"))))
    ) \
    .select("turbine_id", "turbine_name", "capacity", "latitude", "longitude", "region", "region_name")

# 4. Operational Status Dimension Table (operational_status_dim)
# Clé stable : hash des attributs naturels (status + department)
operational_status_dim = df.select("status", "responsible_department").distinct() \
    .withColumn("status_id", 
        F.abs(F.hash(F.concat_ws("_", F.col("status"), F.col("responsible_department"))))
    ) \
    .select("status_id", "status", "responsible_department")

# 5. Location Dimension Table (location_dim)
# Clé stable : hash des coordonnées et région
location_dim = df.select("latitude", "longitude", "region", "region_name").distinct() \
    .withColumn("location_id", 
        F.abs(F.hash(F.concat_ws("_", F.col("region"), F.col("latitude"), F.col("longitude"))))
    ) \
    .select("location_id", "latitude", "longitude", "region", "region_name")

print("Tables de dimension créées avec clés stables (hash des attributs naturels) :")
print(f"- date_dim: {date_dim.count()} lignes")
print(f"- time_dim: {time_dim.count()} lignes")
print(f"- turbine_dim: {turbine_dim.count()} lignes")
print(f"- operational_status_dim: {operational_status_dim.count()} lignes")
print(f"- location_dim: {location_dim.count()} lignes")

Tables de dimension créées avec clés stables (hash des attributs naturels) :
- date_dim: 2 lignes
- time_dim: 144 lignes
- turbine_dim: 3 lignes
- operational_status_dim: 6 lignes
- location_dim: 3 lignes


In [7]:
# Aperçu des tables créées
print("=" * 30)
print("APERÇU DES TABLES DE DIMENSION")
print("=" * 30)

print("\n1. DATE_DIM:")
date_dim.show(5)

print("\n2. TIME_DIM:")
time_dim.show(5)

print("\n3. TURBINE_DIM:")
turbine_dim.show(5)

print("\n4. OPERATIONAL_STATUS_DIM:")
operational_status_dim.show(5)

print("\n5. LOCATION_DIM:")
location_dim.show(5)

APERÇU DES TABLES DE DIMENSION

1. DATE_DIM:
+----------+---+-----+-------+----+
|   date_id|day|month|quarter|year|
+----------+---+-----+-------+----+
|2024-06-15| 15|    6|      2|2024|
|2024-06-16| 16|    6|      2|2024|
+----------+---+-----+-------+----+


2. TIME_DIM:
+-------------------+-----------+--------------+----------------+-----------+
|            time_id|hour_of_day|minute_of_hour|second_of_minute|time_period|
+-------------------+-----------+--------------+----------------+-----------+
|1970-01-01 00:20:00|          0|            20|               0|      Night|
|1970-01-01 13:50:00|         13|            50|               0|  Afternoon|
|1970-01-01 15:20:00|         15|            20|               0|  Afternoon|
|1970-01-01 03:30:00|          3|            30|               0|      Night|
|1970-01-01 11:50:00|         11|            50|               0|    Morning|
+-------------------+-----------+--------------+----------------+-----------+
only showing top 5 row

In [8]:
# Joindre les dimensions au DataFrame principal pour obtenir les clés étrangères
df_with_keys = df \
    .join(turbine_dim.select("turbine_id", "turbine_name", "capacity", "latitude", "longitude", "region", "region_name"),
          ["turbine_name", "capacity", "latitude", "longitude", "region", "region_name"], "left") \
    .join(operational_status_dim.select("status_id", "status", "responsible_department"),
          ["status", "responsible_department"], "left") \
    .join(location_dim.select("location_id", "latitude", "longitude", "region", "region_name"),
          ["latitude", "longitude", "region", "region_name"], "left")

# TABLE DE FAITS 1: Fact_Energy_Production
fact_energy_production = df_with_keys.select(
    col("production_id").alias("fact_production_id"),
    col("date").alias("date_id"),
    col("time").alias("time_id"),
    "turbine_id",
    "status_id",
    "energy_produced",
    "wind_speed_100m",
    "wind_direction"
).select(
    "fact_production_id",
    "date_id",
    "time_id",
    "turbine_id",
    "status_id",
    "energy_produced",
    "wind_speed_100m",
    "wind_direction"
).orderBy("date_id", "time_id", "turbine_id")

# TABLE DE FAITS 2: Fact_Weather_Conditions
fact_weather_conditions = df_with_keys.select(
    col("weather_id").alias("fact_weather_id"),
    col("date").alias("date_id"),
    col("time").alias("time_id"),
    "location_id",
    "temperature_2m",
    "pressure_msl",
    "precipitation",
    "wind_gust_10m",
    "wind_speed_100m"
).select(
    "fact_weather_id",
    "date_id",
    "time_id",
    "location_id",
    "temperature_2m",
    "pressure_msl",
    "precipitation",
    "wind_gust_10m",
    "wind_speed_100m"
).distinct().orderBy("date_id", "time_id", "location_id")

print("\nTables de faits créées:")
print(f"- fact_energy_production: {fact_energy_production.count()} lignes")
print(f"- fact_weather_conditions: {fact_weather_conditions.count()} lignes")


Tables de faits créées:
- fact_energy_production: 864 lignes
- fact_weather_conditions: 864 lignes


In [9]:
print("=" * 26)
print("APERÇU DES TABLES DE FAITS")
print("=" * 26)

print("\n1. FACT_ENERGY_PRODUCTION:")
fact_energy_production.show(5)

print("\n2. FACT_WEATHER_CONDITIONS:")
fact_weather_conditions.show(5)

APERÇU DES TABLES DE FAITS

1. FACT_ENERGY_PRODUCTION:
+------------------+----------+-------------------+----------+---------+---------------+---------------+--------------+
|fact_production_id|   date_id|            time_id|turbine_id|status_id|energy_produced|wind_speed_100m|wind_direction|
+------------------+----------+-------------------+----------+---------+---------------+---------------+--------------+
|              6051|2024-06-15|1970-01-01 00:00:00| 334974073|268321653|        1651.84|          19.80|            NW|
|              6049|2024-06-15|1970-01-01 00:00:00|1091227532|268321653|        1783.39|          15.10|            SW|
|              6050|2024-06-15|1970-01-01 00:00:00|1155570426|595212875|           0.00|          20.20|             E|
|              6054|2024-06-15|1970-01-01 00:10:00| 334974073|268321653|        1459.19|          18.60|             W|
|              6052|2024-06-15|1970-01-01 00:10:00|1091227532|268321653|        1351.88|          15.13| 

In [10]:
# CRÉATION DES TABLES DANS POSTGRESQL (MODE PRODUCTION - IDEMPOTENT)

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
) as conn:
    with conn.cursor() as cur:
        # Créer le schéma
        cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
        
        # Créer les tables de dimension SEULEMENT si elles n'existent pas
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.date_dim (
                date_id DATE PRIMARY KEY,
                day INTEGER NOT NULL,
                month INTEGER NOT NULL,
                quarter INTEGER NOT NULL,
                year INTEGER NOT NULL
            );
        """)
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.time_dim (
                time_id TIME PRIMARY KEY,
                hour_of_day INTEGER NOT NULL,
                minute_of_hour INTEGER NOT NULL,
                second_of_minute INTEGER NOT NULL,
                time_period VARCHAR(20) NOT NULL
            );
        """)
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.turbine_dim (
                turbine_id BIGINT PRIMARY KEY,
                turbine_name VARCHAR(100) NOT NULL,
                capacity INTEGER,
                latitude NUMERIC(9,6),
                longitude NUMERIC(9,6),
                region VARCHAR(50),
                region_name VARCHAR(200)
            );
        """)
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.operational_status_dim (
                status_id BIGINT PRIMARY KEY,
                status VARCHAR(100) NOT NULL,
                responsible_department VARCHAR(100)
            );
        """)
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.location_dim (
                location_id BIGINT PRIMARY KEY,
                latitude NUMERIC(9,6),
                longitude NUMERIC(9,6),
                region VARCHAR(50),
                region_name VARCHAR(200)
            );
        """)
        
        # Créer les tables de faits avec clés étrangères
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.fact_energy_production (
                fact_production_id BIGINT PRIMARY KEY,
                date_id DATE NOT NULL,
                time_id TIME NOT NULL,
                turbine_id BIGINT NOT NULL,
                status_id BIGINT NOT NULL,
                energy_produced NUMERIC(12,2),
                wind_speed_100m NUMERIC(6,2),
                wind_direction VARCHAR(10),
                FOREIGN KEY (date_id) REFERENCES gold.date_dim(date_id),
                FOREIGN KEY (time_id) REFERENCES gold.time_dim(time_id),
                FOREIGN KEY (turbine_id) REFERENCES gold.turbine_dim(turbine_id),
                FOREIGN KEY (status_id) REFERENCES gold.operational_status_dim(status_id)
            );
        """)
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS gold.fact_weather_conditions (
                fact_weather_id BIGINT PRIMARY KEY,
                date_id DATE NOT NULL,
                time_id TIME NOT NULL,
                location_id BIGINT NOT NULL,
                temperature_2m NUMERIC(5,2),
                pressure_msl NUMERIC(7,2),
                precipitation NUMERIC(8,2),
                wind_gust_10m NUMERIC(6,2),
                wind_speed_100m NUMERIC(6,2),
                FOREIGN KEY (date_id) REFERENCES gold.date_dim(date_id),
                FOREIGN KEY (time_id) REFERENCES gold.time_dim(time_id),
                FOREIGN KEY (location_id) REFERENCES gold.location_dim(location_id)
            );
        """)
        
        conn.commit()
        
print("\nDébut de l'insertion des données en mode UPSERT...")


Début de l'insertion des données en mode UPSERT...


In [11]:
# INSERTION DES DIMENSIONS EN MODE UPSERT (PRODUCTION)

print("Insertion des dimensions en mode UPSERT...")

# Préparer time_dim avec format TIME
time_dim_to_save = time_dim.withColumn(
    "time_id", 
    F.to_timestamp(F.date_format(F.col("time_id"), "HH:mm:ss"), "HH:mm:ss")
)

# Convertir en pandas pour faire des UPSERTS avec psycopg
date_dim_pd = date_dim.toPandas()
time_dim_pd = time_dim_to_save.toPandas()
turbine_dim_pd = turbine_dim.toPandas()
operational_status_dim_pd = operational_status_dim.toPandas()
location_dim_pd = location_dim.toPandas()

# UPSERT avec INSERT ON CONFLICT
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
) as conn:
    with conn.cursor() as cur:
        # 1. Date Dimension
        for _, row in date_dim_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.date_dim (date_id, day, month, quarter, year)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (date_id) DO UPDATE SET
                    day = EXCLUDED.day,
                    month = EXCLUDED.month,
                    quarter = EXCLUDED.quarter,
                    year = EXCLUDED.year;
            """, (row['date_id'], row['day'], row['month'], row['quarter'], row['year']))
        print(f"date_dim: {len(date_dim_pd)} lignes insérées/mises à jour")
        
        # 2. Time Dimension
        for _, row in time_dim_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.time_dim (time_id, hour_of_day, minute_of_hour, second_of_minute, time_period)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (time_id) DO UPDATE SET
                    hour_of_day = EXCLUDED.hour_of_day,
                    minute_of_hour = EXCLUDED.minute_of_hour,
                    second_of_minute = EXCLUDED.second_of_minute,
                    time_period = EXCLUDED.time_period;
            """, (row['time_id'], row['hour_of_day'], row['minute_of_hour'], 
                  row['second_of_minute'], row['time_period']))
        print(f"time_dim: {len(time_dim_pd)} lignes insérées/mises à jour")
        
        # 3. Turbine Dimension
        for _, row in turbine_dim_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.turbine_dim (turbine_id, turbine_name, capacity, latitude, longitude, region, region_name)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (turbine_id) DO UPDATE SET
                    turbine_name = EXCLUDED.turbine_name,
                    capacity = EXCLUDED.capacity,
                    latitude = EXCLUDED.latitude,
                    longitude = EXCLUDED.longitude,
                    region = EXCLUDED.region,
                    region_name = EXCLUDED.region_name;
            """, (row['turbine_id'], row['turbine_name'], row['capacity'], 
                  row['latitude'], row['longitude'], row['region'], row['region_name']))
        print(f"turbine_dim: {len(turbine_dim_pd)} lignes insérées/mises à jour")
        
        # 4. Operational Status Dimension
        for _, row in operational_status_dim_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.operational_status_dim (status_id, status, responsible_department)
                VALUES (%s, %s, %s)
                ON CONFLICT (status_id) DO UPDATE SET
                    status = EXCLUDED.status,
                    responsible_department = EXCLUDED.responsible_department;
            """, (row['status_id'], row['status'], row['responsible_department']))
        print(f"operational_status_dim: {len(operational_status_dim_pd)} lignes insérées/mises à jour")
        
        # 5. Location Dimension
        for _, row in location_dim_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.location_dim (location_id, latitude, longitude, region, region_name)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (location_id) DO UPDATE SET
                    latitude = EXCLUDED.latitude,
                    longitude = EXCLUDED.longitude,
                    region = EXCLUDED.region,
                    region_name = EXCLUDED.region_name;
            """, (row['location_id'], row['latitude'], row['longitude'], row['region'], row['region_name']))
        print(f"location_dim: {len(location_dim_pd)} lignes insérées/mises à jour")
        
        conn.commit()

print("\nDimensions chargées avec succès en mode UPSERT (idempotent)")

Insertion des dimensions en mode UPSERT...
date_dim: 2 lignes insérées/mises à jour
time_dim: 144 lignes insérées/mises à jour
turbine_dim: 3 lignes insérées/mises à jour
operational_status_dim: 6 lignes insérées/mises à jour
location_dim: 3 lignes insérées/mises à jour

Dimensions chargées avec succès en mode UPSERT (idempotent)


In [12]:
# INSERTION DES TABLES DE FAITS EN MODE UPSERT (PRODUCTION)

print("\nInsertion des tables de faits en mode UPSERT...")

# Préparer les tables de faits
fact_energy_production_to_save = fact_energy_production.withColumn(
    "time_id", 
    F.to_timestamp(F.date_format(F.col("time_id"), "HH:mm:ss"), "HH:mm:ss")
)

fact_weather_conditions_to_save = fact_weather_conditions.withColumn(
    "time_id", 
    F.to_timestamp(F.date_format(F.col("time_id"), "HH:mm:ss"), "HH:mm:ss")
)

# Convertir en pandas
fact_energy_pd = fact_energy_production_to_save.toPandas()
fact_weather_pd = fact_weather_conditions_to_save.toPandas()

# UPSERT des faits
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
) as conn:
    with conn.cursor() as cur:
        # Fact Energy Production
        for _, row in fact_energy_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.fact_energy_production 
                (fact_production_id, date_id, time_id, turbine_id, status_id, 
                energy_produced, wind_speed_100m, wind_direction)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (fact_production_id) DO UPDATE SET
                    date_id = EXCLUDED.date_id,
                    time_id = EXCLUDED.time_id,
                    turbine_id = EXCLUDED.turbine_id,
                    status_id = EXCLUDED.status_id,
                    energy_produced = EXCLUDED.energy_produced,
                    wind_speed_100m = EXCLUDED.wind_speed_100m,
                    wind_direction = EXCLUDED.wind_direction;
            """, (row['fact_production_id'], row['date_id'], row['time_id'], 
                row['turbine_id'], row['status_id'], row['energy_produced'],
                row['wind_speed_100m'], row['wind_direction']))
        print(f"fact_energy_production: {len(fact_energy_pd)} lignes insérées/mises à jour")
        
        # Fact Weather Conditions
        for _, row in fact_weather_pd.iterrows():
            cur.execute("""
                INSERT INTO gold.fact_weather_conditions 
                (fact_weather_id, date_id, time_id, location_id, 
                temperature_2m, pressure_msl, precipitation, wind_gust_10m, wind_speed_100m)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (fact_weather_id) DO UPDATE SET
                    date_id = EXCLUDED.date_id,
                    time_id = EXCLUDED.time_id,
                    location_id = EXCLUDED.location_id,
                    temperature_2m = EXCLUDED.temperature_2m,
                    pressure_msl = EXCLUDED.pressure_msl,
                    precipitation = EXCLUDED.precipitation,
                    wind_gust_10m = EXCLUDED.wind_gust_10m,
                    wind_speed_100m = EXCLUDED.wind_speed_100m;
            """, (row['fact_weather_id'], row['date_id'], row['time_id'], 
                row['location_id'], row['temperature_2m'], row['pressure_msl'],
                row['precipitation'], row['wind_gust_10m'], row['wind_speed_100m']))
        print(f"fact_weather_conditions: {len(fact_weather_pd)} lignes insérées/mises à jour")
        
        conn.commit()

print("\nCaractéristiques du pipeline:")
print("  - Clés stables (hash des attributs naturels)")
print("  - UPSERT (INSERT ON CONFLICT) - idempotent")
print("  - Pas de doublons, réexécution sécurisée")
print("  - Types PostgreSQL optimisés")
print("  - Clés primaires et étrangères")
print(f"\nRésumé des données:")
print(f"  - {len(date_dim_pd)} dates")
print(f"  - {len(time_dim_pd)} heures")
print(f"  - {len(turbine_dim_pd)} turbines")
print(f"  - {len(operational_status_dim_pd)} statuts opérationnels")
print(f"  - {len(location_dim_pd)} localisations")
print(f"  - {len(fact_energy_pd)} enregistrements de production")
print(f"  - {len(fact_weather_pd)} observations météo")


Insertion des tables de faits en mode UPSERT...
fact_energy_production: 864 lignes insérées/mises à jour
fact_weather_conditions: 864 lignes insérées/mises à jour

Caractéristiques du pipeline:
  - Clés stables (hash des attributs naturels)
  - UPSERT (INSERT ON CONFLICT) - idempotent
  - Pas de doublons, réexécution sécurisée
  - Types PostgreSQL optimisés
  - Clés primaires et étrangères

Résumé des données:
  - 2 dates
  - 144 heures
  - 3 turbines
  - 6 statuts opérationnels
  - 3 localisations
  - 864 enregistrements de production
  - 864 observations météo
